In [2]:
import pandas as pd

ceqr_locations = pd.read_csv("CEQR_Project_Locations.csv")
pluto = pd.read_csv("PLUTO.csv")
zap = pd.read_csv("Zoning_Application_Portal_(ZAP).csv")


In [3]:
zap = zap[(zap["project_status"] == "Active") | (zap["project_status"] == "On-Hold") & (zap["ceqr_number"] != '')]

zap.shape

(335, 11)

In [4]:
merged = pd.merge(zap, ceqr_locations, how="left", left_on="ceqr_number", right_on="CEQR")
print(merged.shape)
merged.head()

(2137, 15)


,project_id,project_name,project_status,public_status,ceqr_type,ceqr_number,borough,current_milestone_date,current_envmilestone_date,app_filed_date,noticed_date,CEQR,Project Name,Block,Lot
0,2026R0338,66 Cromwell Circle,Active,Filed,Type II,27DCP040R,Staten Island,10/08/2026,NaN,10/08/2026,NaN,27DCP040R,66 Cromwell Circle,894.0,121.0
1,2026X0354,2 Ploughmans Bush SNAD,Active,Filed,Type II,26DCP139X,Bronx,06/18/2026,NaN,05/05/2026,NaN,26DCP139X,2 Ploughmans Bush SNAD,5924.0,523.0
2,2026X0354,2 Ploughmans Bush SNAD,Active,Filed,Type II,26DCP139X,Bronx,06/18/2026,NaN,05/05/2026,NaN,26DCP139X,2 Ploughmans Bush SNAD,5924.0,523.0
3,2026M0312,460 Tenth Avenue DIB Certification,Active,Completed,Type II,26DCP105M,Manhattan,03/31/2026,NaN,02/17/2026,NaN,NaN,NaN,NaN,NaN
4,2026M0515,250 Water Street - Renewal,On-Hold,Filed,Type II,27DCP005M,Manhattan,07/29/2026,NaN,07/09/2026,NaN,27DCP005M,250 Water Street - Renewal,98.0,1.0


In [5]:
merged = merged.dropna(subset="CEQR").dropna(subset="Lot").dropna(subset="Block")
merged.shape

(2048, 15)

In [6]:
merged.groupby(by="project_id").count()

,project_name,project_status,public_status,ceqr_type,ceqr_number,borough,current_milestone_date,current_envmilestone_date,app_filed_date,noticed_date,CEQR,Project Name,Block,Lot
project_id,,,,,,,,,,,,,,
2018X0397,3,3,3,3,3,3,3,3,3,3,3,3,3,3
2019K0023,1,1,1,1,1,1,1,0,1,0,1,1,1,1
2019K0147,3,3,3,3,3,3,3,3,3,3,3,3,3,3
2019K0217,1,1,1,1,1,1,1,1,0,1,1,1,1,1
2019K0219,1,1,1,1,1,1,1,0,1,0,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
P2017X0149,5,5,5,5,5,5,5,5,0,0,5,5,5,5
P2018K0011,8,8,8,8,8,8,8,0,8,8,8,8,8,8
P2018K0297,7,7,7,7,7,7,7,7,0,7,7,7,7,7


In [7]:
def get_bbl(borough, block, lot):
    
    bbl = ""
    if borough == "Manhattan":
        bbl = bbl + "1"
    elif borough == "Bronx":
        bbl = bbl + "2"
    elif borough == "Brooklyn":
        bbl = bbl + "3"
    elif borough == "Queens":
        bbl = bbl + "4"
    elif borough == "Staten Island":
        bbl = bbl + "5"

    bbl += block.zfill(5)
    bbl += lot.zfill(4)

    return int(bbl)

In [8]:
import math

bbls = []

for _, row in merged.iterrows():
    borough = row["borough"]
    block = str(int(row["Block"]))
    lot = str(int(row["Lot"]))
    bbl = get_bbl(borough, block, lot)
    bbls.append(bbl)

merged["BBL"] = bbls

merged.head()

,project_id,project_name,project_status,public_status,ceqr_type,ceqr_number,borough,current_milestone_date,current_envmilestone_date,app_filed_date,noticed_date,CEQR,Project Name,Block,Lot,BBL
0,2026R0338,66 Cromwell Circle,Active,Filed,Type II,27DCP040R,Staten Island,10/08/2026,NaN,10/08/2026,NaN,27DCP040R,66 Cromwell Circle,894.0,121.0,5008940121
1,2026X0354,2 Ploughmans Bush SNAD,Active,Filed,Type II,26DCP139X,Bronx,06/18/2026,NaN,05/05/2026,NaN,26DCP139X,2 Ploughmans Bush SNAD,5924.0,523.0,2059240523
2,2026X0354,2 Ploughmans Bush SNAD,Active,Filed,Type II,26DCP139X,Bronx,06/18/2026,NaN,05/05/2026,NaN,26DCP139X,2 Ploughmans Bush SNAD,5924.0,523.0,2059240523
4,2026M0515,250 Water Street - Renewal,On-Hold,Filed,Type II,27DCP005M,Manhattan,07/29/2026,NaN,07/09/2026,NaN,27DCP005M,250 Water Street - Renewal,98.0,1.0,1000980001
5,2026M0432,31 West 52nd Street - Fogo de Chao OAC Renewal,Active,Completed,Type II,26DCP149M,Manhattan,07/07/2026,NaN,05/26/2026,NaN,26DCP149M,31 West 52nd Street - Fogo de Chao OAC Renewal,1268.0,7501.0,1012687501


In [9]:
merged = pd.merge(merged, pluto, left_on=["BBL"], right_on=["BBL"])

merged = merged.dropna(subset="longitude")

merged.shape

(1839, 20)

In [10]:
merged.columns

Index(['project_id', 'project_name', 'project_status', 'public_status',
       'ceqr_type', 'ceqr_number', 'borough', 'current_milestone_date',
       'current_envmilestone_date', 'app_filed_date', 'noticed_date', 'CEQR',
       'Project Name', 'Block', 'Lot', 'BBL', 'Tax block', 'Tax lot',
       'latitude', 'longitude'],
      dtype='object')

In [11]:
cleaned_merge = merged[["project_id", "project_name", "project_status", "ceqr_number", "BBL", "latitude", "longitude"]]

In [12]:
import geopandas as gpd
from shapely.geometry import Point

gdf = gpd.GeoDataFrame(cleaned_merge, geometry=[Point(xy) for xy in zip(cleaned_merge['longitude'], cleaned_merge['latitude'])])

gdf.head()


,project_id,project_name,project_status,ceqr_number,BBL,latitude,longitude,geometry
0,2026R0338,66 Cromwell Circle,Active,27DCP040R,5008940121,40.589533,-74.109019,POINT (-74.10902 40.58953)
1,2026X0354,2 Ploughmans Bush SNAD,Active,26DCP139X,2059240523,40.892563,-73.912062,POINT (-73.91206 40.89256)
2,2026X0354,2 Ploughmans Bush SNAD,Active,26DCP139X,2059240523,40.892563,-73.912062,POINT (-73.91206 40.89256)
3,2026M0515,250 Water Street - Renewal,On-Hold,27DCP005M,1000980001,40.708209,-74.002680,POINT (-74.00268 40.70821)
4,2026M0432,31 West 52nd Street - Fogo de Chao OAC Renewal,Active,26DCP149M,1012687501,40.760925,-73.978013,POINT (-73.97801 40.76092)


In [13]:

geojson_str = gdf.to_json()
print(geojson_str)

with open("ceqr.json", "w") as f:
    f.write(geojson_str)

{"type": "FeatureCollection", "features": [{"id": "0", "type": "Feature", "properties": {"project_id": "2026R0338", "project_name": "66 Cromwell Circle", "project_status": "Active", "ceqr_number": "27DCP040R", "BBL": 5008940121, "latitude": 40.5895329, "longitude": -74.1090191}, "geometry": {"type": "Point", "coordinates": [-74.1090191, 40.5895329]}}, {"id": "1", "type": "Feature", "properties": {"project_id": "2026X0354", "project_name": "2 Ploughmans Bush  SNAD", "project_status": "Active", "ceqr_number": "26DCP139X", "BBL": 2059240523, "latitude": 40.8925631, "longitude": -73.9120623}, "geometry": {"type": "Point", "coordinates": [-73.9120623, 40.8925631]}}, {"id": "2", "type": "Feature", "properties": {"project_id": "2026X0354", "project_name": "2 Ploughmans Bush  SNAD", "project_status": "Active", "ceqr_number": "26DCP139X", "BBL": 2059240523, "latitude": 40.8925631, "longitude": -73.9120623}, "geometry": {"type": "Point", "coordinates": [-73.9120623, 40.8925631]}}, {"id": "3", "t